In [1]:
import os 
# Get the current working directory
current_dir = os.getcwd()
print("Current working directory:", current_dir)

import pandas as pd
import scanpy as sc
import anndata as ad
from tqdm import tqdm
import matplotlib.pyplot as plt # import matplotlib to visualize our qc metrics

# magic incantation to help matplotlib work with our jupyter notebook
%matplotlib inline 

sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')

Current working directory: /gpfs/commons/home/kisaev/Leaflet-analysis/tabula_sapien
scanpy==1.9.3 anndata==0.8.0 umap==0.5.3 numpy==1.23.5 scipy==1.10.1 pandas==1.5.3 scikit-learn==1.0.1 statsmodels==0.13.1 python-igraph==0.10.4 pynndescent==0.5.8


In [2]:
import sys
sys.path.append('../utils')
from functions import * 

In [3]:
# load adata object for tabula muris 
adata = sc.read_h5ad("/gpfs/commons/datasets/controlled/CZI/tabula-sapiens/TS_figshare/TabulaSapiens.h5ad")

In [4]:
adata.obs 

,organ_tissue,method,donor,anatomical_information,n_counts_UMIs,n_genes,cell_ontology_class,free_annotation,manually_annotated,compartment,gender
AAACCCACACTCCTGT_TSP6_Liver_NA_10X_1_1,Liver,10X,TSP6,nan,7633.0,2259,macrophage,Monocyte/Macrophage,True,immune,male
AAACGAAGTACCAGAG_TSP6_Liver_NA_10X_1_1,Liver,10X,TSP6,nan,2858.0,1152,monocyte,Monocyte,True,immune,male
AAACGCTCAACGGCTC_TSP6_Liver_NA_10X_1_1,Liver,10X,TSP6,nan,7787.0,2983,endothelial cell of hepatic sinusoid,Endothelial,True,endothelial,male
AAAGAACAGCCTCTTC_TSP6_Liver_NA_10X_1_1,Liver,10X,TSP6,nan,10395.0,2598,macrophage,Monocyte/Macrophage,True,immune,male
AAAGAACGTAGCACAG_TSP6_Liver_NA_10X_1_1,Liver,10X,TSP6,nan,6610.0,2125,liver dendritic cell,Dendritic cell,True,immune,male
...,...,...,...,...,...,...,...,...,...,...,...
TSP2_Vasculature_aorta_SS2_B114577_B133059_Endothelial_P4_S364,Vasculature,smartseq2,TSP2,aorta,13205.0,579,endothelial cell,endothelial cell,True,endothelial,female
TSP2_Vasculature_aorta_SS2_B114577_B133059_Endothelial_P5_S365,Vasculature,smartseq2,TSP2,aorta,9565.0,529,endothelial cell,endothelial cell,True,endothelial,female
TSP2_Vasculature_aorta_SS2_B114577_B133059_Endothelial_P7_S367,Vasculature,smartseq2,TSP2,aorta,195639.0,2753,endothelial cell,endothelial cell,True,endothelial,female
TSP2_Vasculature_aorta_SS2_B114577_B133059_Endothelial_P8_S368,Vasculature,smartseq2,TSP2,aorta,37260.0,984,endothelial cell,endothelial cell,True,endothelial,female


In [5]:
ss2 = adata.obs[adata.obs["method"] == "smartseq2"]
print(ss2.shape)
# summarize number of ss2 files per donor 
ss2.groupby("donor").size()

(27051, 11)


donor
TSP1     3838
TSP2     9894
TSP3      284
TSP4     3297
TSP5      133
TSP6      568
TSP7     4979
TSP8      633
TSP9       83
TSP10    1495
TSP11    1032
TSP12     277
TSP13     538
TSP14       0
TSP15       0
dtype: int64

In [6]:
# sumamrize number of samples across donors and organs and filter out zero entries 
donors_summ = ss2.groupby(["donor", "organ_tissue"]).size().unstack().fillna(0)
# remove rows that have all zeros
donors_summ = donors_summ.loc[(donors_summ != 0).any(axis=1)]

In [7]:
# filter donors_summ to just row where donor is TSP3
donors_summ.loc[donors_summ.index.str.contains("TSP2")].iloc[0]

organ_tissue
Bladder             447
Blood              1090
Bone_Marrow         720
Eye                   0
Fat                   0
Heart                 0
Kidney              370
Large_Intestine     442
Liver                 0
Lung                901
Lymph_Node         1222
Mammary               0
Muscle             1550
Pancreas              0
Prostate              0
Salivary_Gland        0
Skin                  0
Small_Intestine     606
Spleen              981
Thymus              599
Tongue                0
Trachea             119
Uterus                0
Vasculature         847
Name: TSP2, dtype: int64

In [8]:
sorted(donors_summ.loc[donors_summ.index.str.contains("TSP2")].iloc[0])

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 119,
 370,
 442,
 447,
 599,
 606,
 720,
 847,
 901,
 981,
 1090,
 1222,
 1550]

In [9]:
ss2["bam_file_name"] = ss2.index

/scratch/ipykernel_29488/4141370277.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ss2["bam_file_name"] = ss2.index


In [10]:
# Assuming 'ss2' is your DataFrame
ss2.loc[ss2["donor"] == "TSP1", "donor_id"] = "Pilot1"
ss2.loc[ss2["donor"] == "TSP2", "donor_id"] = "Pilot2"
ss2.loc[ss2["donor"] == "TSP3", "donor_id"] = "Pilot3"
ss2.loc[ss2["donor"] == "TSP4", "donor_id"] = "Pilot4"
ss2.loc[ss2["donor"] == "TSP5", "donor_id"] = "Pilot5"
ss2.loc[ss2["donor"] == "TSP6", "donor_id"] = "Pilot6"
ss2.loc[ss2["donor"] == "TSP7", "donor_id"] = "Pilot7"
ss2.loc[ss2["donor"] == "TSP8", "donor_id"] = "Pilot8"
ss2.loc[ss2["donor"] == "TSP9", "donor_id"] = "Pilot9"
ss2.loc[ss2["donor"] == "TSP10", "donor_id"] = "Pilot10"
ss2.loc[ss2["donor"] == "TSP11", "donor_id"] = "Pilot11"
ss2.loc[ss2["donor"] == "TSP12", "donor_id"] = "Pilot12"
ss2.loc[ss2["donor"] == "TSP13", "donor_id"] = "Pilot13"
ss2.loc[ss2["donor"] == "TSP14", "donor_id"] = "Pilot14"
ss2.loc[ss2["donor"] == "TSP15", "donor_id"] = "Pilot15"

/scratch/ipykernel_29488/2259596608.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ss2.loc[ss2["donor"] == "TSP1", "donor_id"] = "Pilot1"


In [20]:
ss2[ss2["bam_file_name"] == "B107826_P7_S116.homo.gencode.v30.ERCC.chrM"]

,organ_tissue,method,donor,anatomical_information,n_counts_UMIs,n_genes,cell_ontology_class,free_annotation,manually_annotated,compartment,gender,bam_file_name,donor_id
B107826_P7_S116.homo.gencode.v30.ERCC.chrM,Bladder,smartseq2,TSP1,nan,312324.0,3526,bladder urothelial cell,bladder urothelial cell,True,epithelial,female,B107826_P7_S116.homo.gencode.v30.ERCC.chrM,Pilot1


In [18]:
ss2[ss2["donor"] == "TSP1"]

,organ_tissue,method,donor,anatomical_information,n_counts_UMIs,n_genes,cell_ontology_class,free_annotation,manually_annotated,compartment,gender,bam_file_name,donor_id
B107821_A1_S297.homo.gencode.v30.ERCC.chrM,Bladder,smartseq2,TSP1,nan,354874.0,2450,fibroblast,fibroblast,True,stromal,female,B107821_A1_S297.homo.gencode.v30.ERCC.chrM,Pilot1
B107821_A13_S9.homo.gencode.v30.ERCC.chrM,Bladder,smartseq2,TSP1,nan,92917.0,995,vein endothelial cell,vein endothelial cell,True,endothelial,female,B107821_A13_S9.homo.gencode.v30.ERCC.chrM,Pilot1
B107821_A14_S10.homo.gencode.v30.ERCC.chrM,Bladder,smartseq2,TSP1,nan,290701.0,2093,endothelial cell of lymphatic vessel,endothelial cell of lymphatic vessel,True,endothelial,female,B107821_A14_S10.homo.gencode.v30.ERCC.chrM,Pilot1
B107821_A16_S12.homo.gencode.v30.ERCC.chrM,Bladder,smartseq2,TSP1,nan,530456.0,2143,vein endothelial cell,vein endothelial cell,True,endothelial,female,B107821_A16_S12.homo.gencode.v30.ERCC.chrM,Pilot1
B107821_A17_S13.homo.gencode.v30.ERCC.chrM,Bladder,smartseq2,TSP1,nan,2533422.0,1110,plasma cell,plasma cell,True,immune,female,B107821_A17_S13.homo.gencode.v30.ERCC.chrM,Pilot1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
B107920_P22_S283.homo.gencode.v30.ERCC.chrM,Pancreas,smartseq2,TSP1,Endocrine,342511.0,1349,pancreatic ductal cell,pancreatic ductal cell,True,epithelial,female,B107920_P22_S283.homo.gencode.v30.ERCC.chrM,Pilot1
B107920_P3_S264.homo.gencode.v30.ERCC.chrM,Pancreas,smartseq2,TSP1,Endocrine,139924.0,888,pancreatic acinar cell,pancreatic acinar cell,True,epithelial,female,B107920_P3_S264.homo.gencode.v30.ERCC.chrM,Pilot1
B107920_P4_S265.homo.gencode.v30.ERCC.chrM,Pancreas,smartseq2,TSP1,Endocrine,732809.0,1670,pancreatic acinar cell,pancreatic acinar cell,True,epithelial,female,B107920_P4_S265.homo.gencode.v30.ERCC.chrM,Pilot1
B107920_P7_S268.homo.gencode.v30.ERCC.chrM,Pancreas,smartseq2,TSP1,Endocrine,1058668.0,1396,pancreatic ductal cell,pancreatic ductal cell,True,epithelial,female,B107920_P7_S268.homo.gencode.v30.ERCC.chrM,Pilot1


In [16]:
for org in ss2.organ_tissue.unique():
    print(org)

Fat
Skin
Bone_Marrow
Heart
Eye
Mammary
Uterus
Tongue
Muscle
Liver
Trachea
Spleen
Blood
Lymph_Node
Salivary_Gland
Prostate
Pancreas
Bladder
Kidney
Large_Intestine
Lung
Small_Intestine
Thymus
Vasculature


In [ ]:
# save ss2 file as text file for saving BAM files 
ss2_short = ss2[["donor", "organ_tissue", "bam_file_name", "donor_id"]]
print(ss2_short.head())
ss2_short.to_csv("/commons/projects/CZI-tabula-sapiens/SS2_data_processing/ss2_samples.txt", sep="\t", index=False, header=False)

In [ ]:
# check TSP1 files that we have 
#path_bams=/gpfs/commons/datasets/controlled/CZI/tabula-sapiens/AWS_data/alignment-gencode/SS2/Pilot1
#ls $path_bams/*.bam | grep -oP '.*(?=\.)' | grep -oP '[^/]*(?=\.)' | grep -oP '.*(?=\.Aligned)' > /commons/projects/CZI-tabula-sapiens/SS2_data_processing/TSP1_filenames.txt
tsp1_bams=pd.read_csv("/commons/projects/CZI-tabula-sapiens/SS2_data_processing/TSP1_filenames.txt", header=None)
tsp1_bams.columns = ["filename"]
tsp1_bams.head()

In [ ]:
# find the filenames in tsp1_bams that are not in ss2.index 
tsp1_bams[~tsp1_bams.filename.isin(ss2.index)]

In [ ]:
# filter ss2 to index that is B107809_A18_S138.homo.gencode.v30.ERCC.chrM.Aligned.out.sorted.bam 
ss2[ss2.index.isin(tsp1_bams.filename)]